# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and performing basic analysis on the FAIR² dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All references to dataset structure—record sets, fields, and columns—use their schema `@id` fields per best practices for Croissant-based data curation.

### Dataset Source
The dataset source is provided via a Croissant schema at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the Croissant metadata and records from the dataset using `mlcroissant`. The dataset metadata object can be explored to view dataset-level information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object, use appropriate properties
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Published: {meta.date_published}")
print(f"License: {meta.license}")

## 2. Data Overview

Explore available record sets, their `@id`, and structural fields. All dataset schema elements (record sets, fields/columns) are referenced by their `@id` field.

In [ ]:
# List all record sets and their field/column @ids

record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No explicit record_set entries in top-level metadata (likely all data is in the default or implied main table).")
else:
    print("Record Sets:")
    for rs in record_sets:
        print(f"- {rs['@id']}: {rs.get('name','(no name)')}")

# For this dataset, let's enumerate inferred available record_sets from records API:
all_record_set_ids = dataset.record_sets  # exposes discovered record_set @ids
print("\nAvailable record set @ids:")
for i, rsid in enumerate(all_record_set_ids):
    print(f"{i+1}. {rsid}")

# Explore fields/columns in each record set:
for rsid in all_record_set_ids:
    print("\nFields/columns for RecordSet @id:", rsid)
    # meta.record_sets may not contain detailed columns, so we'll peek at first record
    try:
        sample = next(dataset.records(record_set=rsid))
        print([f for f in sample.keys()])
    except Exception as e:
        print(" (Could not preview records, perhaps record set is metadata only or empty)")

## 3. Data Extraction

Load the main tabular data into a DataFrame for further analysis. All references utilize the appropriate record set and field `@id`. The principal (and only) data record set is used below.

In [ ]:
# Extract data from all available record sets into DataFrames, using `@id` values as keys.
import collections

dataframes = collections.OrderedDict()
available_record_set_ids = dataset.record_sets
for rsid in available_record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded RecordSet @id: {rsid} → shape: {df.shape}")
    else:
        print(f"RecordSet @id: {rsid}: No records found.")

# Display columns/fields for the main record set
main_rsid = next(iter(dataframes.keys()))  # pick the first/main (likely only) data table
print(f"\nColumns for main RecordSet @id='{main_rsid}':")
print(list(dataframes[main_rsid].columns))
dataframes[main_rsid].head()

## 4. Exploratory Data Analysis (EDA)

Here we demonstrate numeric field filtering, normalization, and grouping as basic processing steps. All field references use their schema `@id` keys as columns. You can extend this section for further analysis as needed.

### Select Numeric Fields and a Group Field
Below, we demonstrate these steps for the field `schema:age` (if present), grouping by `schema:sex` (if present). Adjust `numeric_field_id` and `group_field_id` as appropriate.

In [ ]:
# Identify available numeric fields and candidate group fields
df = dataframes[main_rsid]
numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or col.lower().endswith('age') or col.lower().endswith('interval')]
print("Numeric field candidates:", numeric_field_candidates)

# For demonstrative filtering, select the first numeric field that looks like 'age' or 'interval'
numeric_field_id = None
for col in numeric_field_candidates:
    if 'age' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None and numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]

if numeric_field_id:
    print(f"\nSelected numeric field '@id': {numeric_field_id}")
else:
    print("No numeric field available for analysis. Skipping EDA.")

group_field_candidates = [col for col in df.columns if 'sex' in col.lower() or 'location' in col.lower() or df[col].dtype == 'object']
group_field_id = group_field_candidates[0] if group_field_candidates else None
if group_field_id:
    print(f"\nSelected group field '@id': {group_field_id}")

if numeric_field_id:
    # Remove missing or invalid data
    filtered_df = df[df[numeric_field_id].notnull()]
    if filtered_df[numeric_field_id].dtype == 'O':
        # Try to convert to numeric, coercing errors
        filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
        filtered_df = filtered_df[filtered_df[numeric_field_id].notnull()]

    # Demonstrate thresholding (e.g., records with age > 50)
    threshold = 50
    filtered_above_threshold = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_above_threshold.head())

    # Normalize the numeric field (z-score)
    filtered_above_threshold[numeric_field_id + '_normalized'] = (
        (filtered_above_threshold[numeric_field_id] - filtered_above_threshold[numeric_field_id].mean()) /
        filtered_above_threshold[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_above_threshold[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Group by group_field if available
    if group_field_id in filtered_above_threshold.columns:
        grouped = filtered_above_threshold.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped.head())

## 5. Visualization

Visualize the distribution of the numeric field and group-wise statistics (if present). Adjust fields used here to match your analysis focus.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for numeric field
if numeric_field_id:
    plt.figure(figsize=(7,5))
    sns.histplot(df[numeric_field_id].dropna().astype(float), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Boxplot grouped by group_field_id
if numeric_field_id and group_field_id in df.columns:
    plt.figure(figsize=(9,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, showmeans=True)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

We have demonstrated how to load and examine the FAIR² colorectal cancer dataset using the `mlcroissant` library, referenced by schema `@id` for strict data provenance. The notebook displayed available record sets, explored main fields, performed filtering and normalization on a numeric clinical variable, and visualized value distributions and group differences.

This FAIR-adherent workflow enables portable, reproducible data exploration and lays the groundwork for further domain-specific analysis, such as biomarker stratification, anatomical predication, or clinical outcome modeling using the Croissant framework.